# ARC-AGI-2: Qwen3.8-27B-FP8 + Python tools, offline on 4xL4

Generated by `kaggle/build_notebook.py --variant v2` from the team repo (the repo is the source of truth; do not
edit this notebook by hand).

**What it does**
- **Mode switch.** On the competition rerun (`KAGGLE_IS_COMPETITION_RERUN`, or a test file whose task ids differ
  from the public one) it solves the hidden test set. Otherwise (commit or interactive run) it solves a small local
  test set with known solutions (`chankhavu/arc2-local-testcases`), scores itself, and the last cell fails the run
  if that validation did not work. Both modes write a valid `submission.json` for the competition test file first.
- **Model and server.** Qwen3.8-27B-FP8 (hash-verified mirror) on vLLM 0.23.0 from the user's own offline wheel
  source (`chankhavu/vllm-0-23-0-py312-cu130`): TP=4, FP8 KV cache with the Triton attention backend, prefix caching.
- **Agent.** E012 tool agent (`agent_qwen.solve`): raw `/v1/completions` with the Qwen template, `run_python` and
  `submit_answer` tools in a local child-process executor, budget notes, forced `submit_answer` at the cap.
- **Scheduler.** One attempt per test pair, budget = min(48k, remaining capacity / remaining pairs); a second attempt
  only with spare capacity. Streams are cut at the deadline, which forces the answer.
- **Time limits (from the first cell).** Force answers at 11h00m, hard stop 11h30m, in-solver emergency writer at
  11h42m, notebook guard kills the solver process group at 11h45m; the last cell validates `submission.json` and
  repairs any slot. If installation or server start is slow, per-pair budgets shrink accordingly.


## 1. Configuration

The wall clock for every deadline starts here (`ARC_T0`).

In [ ]:
import os, time
os.environ['ARC_T0'] = str(time.time())  # deadline clock starts at the first cell
os.environ['ARC_MODE'] = 'submit' if os.environ.get('KAGGLE_IS_COMPETITION_RERUN') else 'local'
SETTINGS = {
 "ARC_SERVER": "vllm",
 "ARC_MTP": 0,
 "ARC_KV_PER_SEQ": 40000,
 "ARC_RATE_PRIOR": 220,
 "ARC_LOCAL_TASKS": "1818057f,2ba387bc,dd6b8c4b,58f5dbd5",
 "ARC_LOCAL_READY_MIN": 6,
 "ARC_LOCAL_HARD_MIN": 2
}
for k, v in SETTINGS.items():
    os.environ.setdefault(k, str(v))
os.makedirs('/kaggle/working/arc_solver/evals/arc_harness', exist_ok=True)
os.makedirs('/kaggle/working/arc_solver/notes/prompts', exist_ok=True)
print('mode', os.environ['ARC_MODE'], {k: v for k, v in os.environ.items() if k.startswith('ARC_')})

## 2. Modules

The E012 tool harness (`evals/arc_harness/`), the ARC Prize prompt, and the solver (server install/launch, scheduler, deadlines, submission writer) are written to `/kaggle/working/arc_solver/` with their repo layout.

In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/__init__.py
"""ARC-AGI-2 agent harness: prompts, model access, grid checks (more layers to come; see notes/specs)."""


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/grids.py
"""ARC grid validity: the single definition used for answers, submissions and scoring."""


def valid_grid(grid):
    """A rectangular list of 1-30 rows of 1-30 Python ints in 0-9."""
    return (isinstance(grid, list) and 1 <= len(grid) <= 30
            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)
            and len({len(row) for row in grid}) == 1
            and all(type(value) is int and 0 <= value <= 9 for row in grid for value in row))


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/arcprize.py
"""ARC Prize public-eval prompt and answer parser (github.com/arcprize/arc-agi-benchmarking), verbatim."""
import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
TASKS = ROOT / "data/arc-agi-2-github/data/evaluation"
TEMPLATE = (ROOT / "notes/prompts/arcprize_official_system_prompt.txt").read_text()


def build_prompt(train, test_input):
    ex = ""
    for i, pair in enumerate(train):
        ex += f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(pair["input"]) + "\n\n"
        ex += "OUTPUT: \n\n" + json.dumps(pair["output"]) + "\n\n"
    return TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))


def is_grid(x):
    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)


def extract_from_boxed(s):
    m = re.search(r"\\boxed\{(.*?)\}", s, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            return g if is_grid(g) else None
        except json.JSONDecodeError:
            return None
    return None


def backscan_json_parser(s):
    end = max(s.rfind("]"), s.rfind("}"))
    if end < 0:
        return None
    close = s[end]
    open_ = "[" if close == "]" else "{"
    depth = 1
    for i in range(end - 1, -1, -1):
        depth += (s[i] == close) - (s[i] == open_)
        if depth == 0:
            try:
                g = json.loads(s[i:end + 1])
                return g if is_grid(g) else None
            except json.JSONDecodeError:
                return None
    return None


def parse(s):
    return extract_from_boxed(s) or backscan_json_parser(s or "")



In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/prompts.py
"""Everything the model reads: the opening message, tool descriptions and fixed instructions.

Kept byte-identical to harness persistent-v3 so results stay comparable across refactors.
"""
from .arcprize import build_prompt

# Optional suffixes appended to the ARC Prize prompt (--prompt). "plain" keeps the original prompt.
PROMPT_SUFFIXES = {
    "plain": "",
    "lean": ("\n\nYou have a Python tool. Work efficiently: inspect the grids with code instead of reading them "
             "by hand, form a hypothesis, and write code that computes the output. Never construct, copy or count "
             "grids by hand. Check your rule on every training pair with code before answering, and if a pair "
             "fails, fix the rule rather than explaining the mismatch away. Keep your reasoning brief."),
    "lean2": ("\n\nYou have a Python tool. The variables `train` (list of {'input', 'output'} grids) and "
              "`test_input` are already loaded in it: use them directly and never retype or reassign them. "
              "Work efficiently: inspect the grids with code instead of reading them by hand, form a hypothesis, "
              "and write code that computes the output. Never construct, copy or count grids by hand. Check your "
              "rule on every training pair with code before answering, and if a pair fails, fix the rule rather "
              "than explaining the mismatch away. Before answering, check that the rule's assumptions also hold "
              "on test_input. Keep your reasoning brief."),
}
CODE_MODE_SUFFIX = "\n\nUse run_python to investigate; submit self-contained transform(grid) code with submit_solution."
FINALIZE = "Finalize now. Return only your best test output as a JSON grid."
MODES = ("grid", "code")


def initial_messages(train, test_input, prompt, mode):
    content = build_prompt(train, test_input) + PROMPT_SUFFIXES[prompt]
    if mode == "code":
        content += CODE_MODE_SUFFIX
    return [{"role": "user", "content": content}]


def tool_specs(mode, cell_timeout):
    descriptions = {"run_python": (
        "Execute a Python cell in your persistent IPython session. Variables/functions persist. "
        "Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. "
        f"scipy is available. No internet or host files. Timeout {cell_timeout}s. "
        "Use print() to inspect results. Check transformations on all training pairs."
    )}
    if mode == "code":
        descriptions["submit_solution"] = (
            "Submit self-contained Python code defining transform(grid), which returns a rectangular "
            "grid of ints 0-9 (or numpy array). Include imports/helpers; execution uses a fresh namespace. "
            "The harness checks every training example. Failures return feedback so you can revise. "
            "If all pass, it applies transform to test_input and ends this attempt."
        )
    return [{"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}},
                       "required": ["code"], "additionalProperties": False},
    }} for name, description in descriptions.items()]


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/executor.py
"""Code-execution backends for the tool harness, behind one small interface.

An executor is a persistent Python session for one attempt:

    with make_executor(train, test_input) as ex:     # train: [{'input', 'output'}], test_input: grid
        result = ex.execute(code)                    # {'process_status': completed|error|timeout,
                                                     #  'stdout': str, 'stderr': str, ['state_reset': True]}

It preloads `train`, `test_input`, `np` and `json`, keeps variables between cells, and shows the value of a
cell's last expression (IPython-style). Attributes used by the answer helpers: `train`, `test_input`, `max_output`.

Backends:
- `SandboxSession` (evals/sandbox_client.py): the Docker NeMo-Skills sandbox (no network, no host mounts).
- `LocalExecutor` (here): a child Python process per attempt, for Kaggle where Docker is unavailable. It is not a
  security boundary (same user, same filesystem). It adds limits (address space, cumulative CPU, file size, nice,
  oom_score_adj=1000), a per-cell timeout (SIGINT to the process group, then kill and restart) and output truncation.

`salvage_answer` (qwen-tools-3) recovers an answer from the session's own functions when the model left none.
`submit_answer` / `check_answer_on_train` / `tool_output` are kept identical to evals/run_deepseek_tools.py and
evals/sandbox_client.py (copied, so this package runs without those scripts).
"""
import json
import os
import select
import signal
import subprocess
import sys
import time
from uuid import uuid4

from .grids import valid_grid

# ---------------------------------------------------------------- local executor

_CHILD = r'''
import ast, contextlib, io, json, os, resource, signal, sys, traceback
resp = os.fdopen(int(sys.argv[1]), "w")
limit = int(sys.argv[2])
mem, cpu, fsize, nice = (int(x) for x in sys.argv[3:7])
# Self-imposed limits (no preexec_fn: the parent spawns from many threads). The child leads its own session.
if mem:
    resource.setrlimit(resource.RLIMIT_AS, (mem, mem))
if cpu:
    resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu + 5))
if fsize:
    resource.setrlimit(resource.RLIMIT_FSIZE, (fsize, fsize))
    signal.signal(signal.SIGXFSZ, signal.SIG_IGN)  # oversized writes fail with OSError instead of killing us
if nice:
    os.nice(nice)
try:
    with open("/proc/self/oom_score_adj", "w") as f:
        f.write("1000")  # under host memory pressure, kill tool code before the model server
except OSError:
    pass
ns = {"__name__": "__main__"}

def run(code):
    tree = ast.parse(code, "<cell>", "exec")
    last = None
    if tree.body and isinstance(tree.body[-1], ast.Expr):
        last = ast.Expression(tree.body.pop().value)
    exec(compile(tree, "<cell>", "exec"), ns)
    if last is not None:
        value = eval(compile(last, "<cell>", "eval"), ns)
        if value is not None:
            print(repr(value))

def plain_traceback(exc):
    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename != "<string>"]
    text = "".join(traceback.format_list(frames)) if frames else ""
    return "Traceback (most recent call last):\n" + text + "".join(traceback.format_exception_only(type(exc), exc))

for line in sys.stdin:
    req = json.loads(line)
    out, err = io.StringIO(), io.StringIO()
    status = "completed"
    try:
        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
            run(req["code"])
    except KeyboardInterrupt:
        status = "timeout"
        err.write("KeyboardInterrupt: cell timed out (state kept)\n")
    except BaseException as exc:
        status = "error"
        try:
            err.write(plain_traceback(exc))
        except BaseException:
            err.write(f"{type(exc).__name__}: {exc}\n")
    o, e = out.getvalue(), err.getvalue()
    if len(o) > limit:
        o = o[:limit] + "\n[output truncated]"
    if len(e) > limit:
        e = e[:limit] + "\n[output truncated]"
    resp.write(json.dumps({"process_status": status, "stdout": o, "stderr": e}) + "\n")
    resp.flush()
'''


class LocalExecutor:
    """Persistent Python in a child process. On timeout the cell gets SIGINT (state kept, like the patched
    sandbox); if it does not stop within `grace` seconds the process is killed and restarted with only the
    task data (`state_reset`)."""

    def __init__(self, train, test_input, timeout=30, max_output=16000, memory_mb=4096, grace=5,
                 python=sys.executable, cpu_seconds=1200, fsize_mb=64, nice=10, start_timeout=90):
        self.train, self.test_input = train, test_input
        self.timeout, self.max_output, self.memory_mb, self.grace = timeout, max_output, memory_mb, grace
        self.python, self.cpu_seconds, self.fsize_mb, self.nice = python, cpu_seconds, fsize_mb, nice
        self.start_timeout = start_timeout  # the prelude (numpy import) may be slow on a busy 4-CPU box
        self.proc = self.resp = None
        self.closed = False
        data = json.dumps({"train": train, "test_input": test_input})
        self.prelude = (
            "import json, numpy as np\n"
            f"_arc_data = json.loads({data!r})\n"
            "train, test_input = _arc_data['train'], _arc_data['test_input']\n"
            "del _arc_data\n"
        )

    def _spawn_once(self, timeout):
        r, w = os.pipe()
        self.proc = subprocess.Popen(
            [self.python, "-I", "-u", "-c", _CHILD, str(w), str(self.max_output), str(self.memory_mb * 2 ** 20),
             str(int(self.cpu_seconds)), str(self.fsize_mb * 2 ** 20), str(self.nice)],
            stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, pass_fds=(w,),
            start_new_session=True, text=True, cwd="/tmp",
            env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
                 "MKL_NUM_THREADS": "1"})
        os.close(w)
        self.resp, self.buf = r, b""
        return self._run(self.prelude, timeout)

    def _spawn(self):
        """Start the child and load the task data; one retry with a longer timeout, else RuntimeError."""
        for timeout in (max(self.timeout, 30), self.start_timeout):
            result = self._spawn_once(timeout)
            if result is not None and result["process_status"] == "completed":
                return
            self._kill()
        detail = "" if result is None else result.get("stderr", "")[:500]
        raise RuntimeError(f"Local executor initialization failed {detail}".strip())

    def _read(self, deadline):
        while b"\n" not in self.buf:
            wait = deadline - time.monotonic()
            if wait <= 0:
                return None
            ready, _, _ = select.select([self.resp], [], [], wait)
            if ready:
                chunk = os.read(self.resp, 1 << 16)
                if not chunk:
                    return "eof"
                self.buf += chunk
        line, self.buf = self.buf.split(b"\n", 1)
        return json.loads(line)

    def _signal_group(self, sig):
        try:
            os.killpg(self.proc.pid, sig)  # the child leads its own session: this also reaches grandchildren
        except (ProcessLookupError, PermissionError, OSError):
            try:
                self.proc.send_signal(sig)
            except (ProcessLookupError, OSError):
                pass

    def _kill(self):
        if self.proc is not None:
            self._signal_group(signal.SIGKILL)
            try:
                self.proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            try:
                self.proc.stdin.close()
            except Exception:
                pass
        if self.resp is not None:
            os.close(self.resp)
        self.proc = self.resp = None

    def _run(self, code, timeout=None):
        """Run one cell; returns the result, or None if the process died or had to be killed."""
        self.last_failure = "error"
        try:
            self.proc.stdin.write(json.dumps({"code": code}) + "\n")
            self.proc.stdin.flush()
        except (BrokenPipeError, OSError):
            return None
        result = self._read(time.monotonic() + (timeout or self.timeout))
        if result is None:  # timed out: interrupt, keep state if the cell stops
            self._signal_group(signal.SIGINT)
            self.last_failure = "timeout"
            result = self._read(time.monotonic() + self.grace)
            if isinstance(result, dict):
                result["process_status"] = "timeout"
                return result
            return None
        return result if isinstance(result, dict) else None

    def initialize(self):
        self._spawn()

    def execute(self, code):
        if self.closed:
            raise RuntimeError("Executor is closed")
        if self.proc is None:
            self._spawn()
        result = self._run(code)
        if result is None:
            # Died (memory or CPU limit, os._exit, segfault) or ignored SIGINT: restart with task data only.
            self._kill()
            try:
                self._spawn()
            except RuntimeError as exc:  # leave proc None: the next execute tries to start again
                return {"process_status": "error", "stdout": "", "state_reset": True,
                        "stderr": f"Python process died and could not be restarted: {exc}"}
            result = {"process_status": self.last_failure, "stdout": "", "state_reset": True,
                      "stderr": ("Cell did not stop after the timeout; process restarted." if self.last_failure == "timeout"
                                 else "Python process died (crash, exit or memory limit); process restarted.")}
        return result

    def close(self):
        if not self.closed:
            self.closed = True
            self._kill()

    def __enter__(self):
        try:
            self.initialize()
        except BaseException:
            self.close()
            raise
        return self

    def __exit__(self, *exc):
        self.close()


def local_factory(timeout=30, max_output=16000, memory_mb=4096, **kwargs):
    """kwargs: grace, cpu_seconds, fsize_mb, nice, start_timeout, python (see LocalExecutor)."""
    return lambda train, test_input: LocalExecutor(train, test_input, timeout, max_output, memory_mb, **kwargs)


def sandbox_factory(url, timeout=30, max_output=16000):
    """Docker sandbox sessions (evals/sandbox_client.py must be importable, i.e. evals/ on sys.path)."""
    from sandbox_client import SandboxSession
    return lambda train, test_input: SandboxSession(url, train, test_input, timeout, max_output)


# ---------------------------------------------------------------- helpers shared with run_deepseek_tools.py

def tool_output(result, limit=16000):
    text = result.get("stdout", "")
    if result.get("stderr"):
        text += "\n[stderr]\n" + result["stderr"]
    if result.get("state_reset"):
        text = "[Python state lost. train/test_input/np restored; redefine other variables.]\n" + text
    text = f"[{result['process_status']}]\n" + (text or "[no output; use print()]")
    return text if len(text) <= limit else text[:limit] + "\n[output truncated]"


def submit_answer(session, code):
    """Evaluate the model's answer expression in its own session and return (feedback, grid|None).

    `code` is an expression (e.g. solve(test_input)) or statements that set `answer`."""
    marker = f"ARC_ANSWER_{uuid4().hex}:"
    cell = f'''
def _arc_answer():
    import contextlib, io, json
    _src = {code!r}
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            _expr = compile(_src, '<answer>', 'eval')
        except SyntaxError:
            _expr = None
        if _expr is not None:
            value = eval(_expr, globals())
        else:
            exec(compile(_src, '<answer>', 'exec'), globals())
            if 'answer' not in globals():
                raise NameError('code did not define answer')
            value = globals()['answer']
        if hasattr(value, 'tolist'):
            value = value.tolist()
        text = json.dumps(value, default=int)
    print({marker!r} + text)
try:
    _arc_answer()
finally:
    del _arc_answer
'''
    result = session.execute(cell)
    grid = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    grid = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
        else:
            return "[answer not accepted] no value returned", None
    else:
        return tool_output(result, session.max_output)[:2000] + "\n[answer not accepted]", None
    if not valid_grid(grid):
        shape = (f"{len(grid)} rows, row lengths {sorted({len(r) for r in grid if isinstance(r, list)})}"
                 if isinstance(grid, list) else type(grid).__name__)
        return ("[answer not accepted] must be a rectangular 1-30 by 1-30 grid of ints 0-9; got " + shape), None
    return f"[answer accepted: {len(grid)}x{len(grid[0])} grid]", grid


def check_answer_on_train(session, code):
    """Re-evaluate the submitted expression with `test_input` bound to each training input and compare with the
    host's training outputs. None when there is nothing to check, else (failed_indices, feedback_text)."""
    try:
        expr = compile(code, "<answer>", "eval")
    except SyntaxError:
        return None
    if "test_input" not in expr.co_names:
        return None
    marker = f"ARC_CHECK_{uuid4().hex}:"
    inputs = [pair["input"] for pair in session.train]
    cell = f'''
def _arc_check():
    import contextlib, io, json
    _expr = compile({code!r}, '<answer>', 'eval')
    outs = []
    with contextlib.redirect_stdout(io.StringIO()):
        for _inp in json.loads({json.dumps(inputs)!r}):
            try:
                v = eval(_expr, {{**globals(), 'test_input': _inp}})
                if hasattr(v, 'tolist'):
                    v = v.tolist()
                outs.append(json.loads(json.dumps(v, default=int)))
            except Exception as exc:
                outs.append({{'error': type(exc).__name__ + ': ' + str(exc)[:200]}})
    print({marker!r} + json.dumps(outs))
try:
    _arc_check()
finally:
    del _arc_check
'''
    result = session.execute(cell)
    outs = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    outs = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
    if not isinstance(outs, list) or len(outs) != len(inputs):
        return None
    failed, notes = [], []
    for i, (o, pair) in enumerate(zip(outs, session.train)):
        if o == pair["output"]:
            continue
        failed.append(i)
        if isinstance(o, dict) and "error" in o:
            notes.append(f"pair {i}: {o['error']}")
        elif valid_grid(o):
            exp = pair["output"]
            if (len(o), len(o[0])) != (len(exp), len(exp[0])):
                notes.append(f"pair {i}: shape {len(o)}x{len(o[0])}, expected {len(exp)}x{len(exp[0])}")
            else:
                notes.append(f"pair {i}: {sum(a != b for ra, rb in zip(o, exp) for a, b in zip(ra, rb))} cells differ")
        else:
            notes.append(f"pair {i}: not a valid grid")
    return failed, "; ".join(notes)


def salvage_answer(session, max_functions=8):
    """No answer was submitted or set: score the session's own one-argument functions on the training inputs
    (host-side comparison) and apply the best one to test_input. Returns (info, grid|None); info carries the
    function name and how many training pairs it reproduced. Costs no model tokens."""
    marker = f"ARC_SALVAGE_{uuid4().hex}:"
    inputs = [pair["input"] for pair in session.train] + [session.test_input]
    cell = f'''
def _arc_salvage():
    import contextlib, copy, inspect, io, json
    import numpy as _np
    fns = []
    for _name, _f in list(globals().items())[::-1]:
        if _name.startswith('_') or not inspect.isfunction(_f) or _f.__module__ != '__main__':
            continue
        try:
            _req = [p for p in inspect.signature(_f).parameters.values()
                    if p.default is p.empty and p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)]
        except (TypeError, ValueError):
            continue
        if len(_req) == 1:
            fns.append((_name, _f))
        if len(fns) >= {max_functions}:
            break
    res = {{}}
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        for _name, _f in fns:
            outs = []
            for _inp in json.loads({json.dumps(inputs)!r}):
                v = None
                for arg in (copy.deepcopy(_inp), _np.array(_inp)):
                    try:
                        v = _f(arg)
                        if hasattr(v, 'tolist'):
                            v = v.tolist()
                        v = json.loads(json.dumps(v, default=int))
                        break
                    except Exception:
                        v = None
                outs.append(v)
            res[_name] = outs
    print({marker!r} + json.dumps(res))
try:
    _arc_salvage()
finally:
    del _arc_salvage
'''
    result = session.execute(cell)
    found = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    found = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
    if not isinstance(found, dict):
        return {"status": result["process_status"]}, None
    best = None  # (train pairs matched, newest first) -> name, grid
    for rank, (name, outs) in enumerate(found.items()):
        if not isinstance(outs, list) or len(outs) != len(inputs) or not valid_grid(outs[-1]):
            continue
        score = sum(o == pair["output"] for o, pair in zip(outs, session.train))
        if best is None or score > best[0]:
            best = (score, name, outs[-1])
    if best is None:
        return {"functions": list(found)}, None
    return {"function": best[1], "train_matched": best[0], "train_pairs": len(session.train)}, best[2]


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/agent_qwen.py
"""Qwen3.8 tool-use agent for ARC-AGI-2 over a raw text-completion API (E012).

Backend-agnostic: the model is any `LLM` with `async complete(prompt, max_tokens, stop) -> Completion`
(`OpenAICompletionsLLM` covers SGLang / vLLM `/v1/completions`; an offline vLLM engine can implement the same
method). Code runs in an `executor.py` executor (Docker sandbox here, `LocalExecutor` on Kaggle).

Why raw completions instead of chat + server-side tool parsing:
- the prompt only ever grows (every past turn is appended verbatim), so the server's prefix cache hits on every
  turn; the Qwen template keeps the thinking of all turns after the last real user message anyway, so this is
  exactly what the chat template would render (checked by run_qwen_tools.py --verify-template);
- a turn cut by max_tokens, or an EOS in the middle of thinking, is continued in place;
- tool calls are parsed here (`parse_tool_calls`), so parse failures are visible and get feedback;
- no dependence on a particular server's tool / reasoning parser.

Harness logic follows E009 sub80s40 (DeepSeek's best): grid mode, `lean2sub` prompt, `run_python` +
`submit_answer` tools, budget note on every tool output with a soft target and hard cap, and at the hard cap a
forced `submit_answer` call (the model only writes the expression) with the session `answer` as fallback.
Exactly one scored grid per attempt; `answer_source` is submit | forced | parse | lenient | session | salvage | None.
"""
import asyncio
import json
import re
import threading
import time
from dataclasses import asdict, dataclass

from .arcprize import build_prompt, parse
from .executor import check_answer_on_train, salvage_answer, submit_answer, tool_output
from .grids import valid_grid
from .prompts import PROMPT_SUFFIXES

HARNESS_VERSION = "qwen-tools-3"  # 2: forced-submit text names the expression; 3: robustness (R2/R3/R7/R8, S4/S7/S8)
CONTEXT = 262144
THINK_CLOSE = "</think>"
GEN_PROMPT = "<|im_start|>assistant\n<think>\n"
TIER_SYSTEM = {
    "xhigh": "Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, "
             "consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final "
             "answer.",
    "medium": None,
    "low": "Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion "
           "without unnecessary elaboration.",
}
# E009 lean2sub: lean2 + the submit_answer hand-off (byte-identical to run_deepseek_tools.py).
_SUBMIT = ("\n\nWhen your code reproduces every training output and its assumptions hold on test_input, call "
           "submit_answer with an expression such as `solve(test_input)`: the harness evaluates it in your session "
           "and that grid is your answer, so never retype the grid. Keep your current best guess for the test "
           "output in a variable `answer`; if you run out of budget it is submitted automatically.")
PROMPTS = {"lean2sub": PROMPT_SUFFIXES["lean2"] + _SUBMIT}
TOOLS_HEADER = ("# Tools\n\nYou have access to the following functions:\n\n<tools>")
TOOLS_FOOTER = (
    "\n</tools>\n\nIf you choose to call a function ONLY reply in the following format with NO suffix:\n\n"
    "<tool_call>\n<function=example_function_name>\n<parameter=example_parameter_1>\nvalue_1\n</parameter>\n"
    "<parameter=example_parameter_2>\nThis is the value for the second parameter\nthat can span\nmultiple lines\n"
    "</parameter>\n</function>\n</tool_call>\n\n<IMPORTANT>\nReminder:\n- Function calls MUST follow the specified "
    "format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags\n"
    "- Required parameters MUST be specified\n- You may provide optional reasoning for your function call in "
    "natural language BEFORE the function call, but NOT after\n- If there is no function call available, answer "
    "the question like normal with your current knowledge and do not tell the user about function calls\n"
    "</IMPORTANT>")
# At the hard cap: close the thinking and open a submit_answer call; the model writes only the expression.
FORCE_THINK = ("\n\nI am out of budget, so I must submit my best answer now as one short expression: `answer` if I "
               "set it, otherwise solve(test_input) or the name of my best function applied to test_input.\n</think>\n\n")
FORCE_CALL = "<tool_call>\n<function=submit_answer>\n<parameter=code>\n"
LOW_YIELD = 20


@dataclass
class AgentConfig:
    tier: str = "xhigh"
    prompt: str = "lean2sub"
    max_total_output: int = 80000   # hard cap: generated tokens per attempt, all turns
    budget_soft: int = 40000        # soft target stated in the budget notes (0 = none)
    budget_notes: float = 0.01      # notes on tool outputs once this fraction of the hard cap is used (0 = off)
    budget_warn: float = 0.75       # from this fraction on, the note says to submit now
    soft_wording: str = "gentle"    # past the soft target: gentle = submit once a rule fits; strong = submit now (E010)
    chunk_tokens: int = 16384       # max tokens per completion request (a turn may take several)
    max_turns: int = 60
    tool_timeout: float = 30
    max_tool_output: int = 16000
    verify_submit: bool = False     # E011: re-run the submitted expression on train inputs; reject once on mismatch
    force_submit_tokens: int = 256  # tokens for the forced submit_answer expression at the cap (0 = off)
    temperature: float = 1.0
    top_p: float = 0.95
    top_k: int = 20
    max_eos_continues: int = 4      # EOS inside thinking: continue the turn at most this many times
    context: int = CONTEXT          # the server's context length (max_model_len); Kaggle runs 131072
    salvage: bool = True            # no answer at the end: apply the session's best-scoring function (qwen-tools-3)
    exec_margin: float = 35         # hard wall-clock cap per executor call = tool_timeout + exec_margin


# ---------------------------------------------------------------- rendering (Qwen3.8 chat template, tools on)

def tool_specs(cfg):
    descriptions = {
        "run_python": (
            "Execute a Python cell in your persistent IPython session. Variables/functions persist. "
            "Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. "
            f"scipy is available. No internet or host files. Timeout {cfg.tool_timeout:g}s. "
            "Use print() to inspect results. Check transformations on all training pairs."),
        "submit_answer": (
            "End this attempt with your test output. `code` is a Python expression evaluated in your session, "
            "e.g. solve(test_input) or answer, giving a rectangular grid of ints 0-9 (list or numpy array); "
            "statements that set a variable `answer` also work. The harness reads the grid directly: do not "
            "retype it. Invalid grids return an error and the attempt continues."),
    }
    return [{"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}},
                       "required": ["code"], "additionalProperties": False},
    }} for name, description in descriptions.items()]


def render_system(tier, tools):
    s = "<|im_start|>system\n"
    if TIER_SYSTEM[tier]:
        s += TIER_SYSTEM[tier] + "\n\n"
    s += TOOLS_HEADER + "".join("\n" + json.dumps(t, ensure_ascii=False) for t in tools) + TOOLS_FOOTER
    return s + "<|im_end|>\n"


def render_user(content):
    return f"<|im_start|>user\n{content.strip()}<|im_end|>\n"


def render_tool_responses(outputs):
    return "<|im_start|>user" + "".join(f"\n<tool_response>\n{o.strip()}\n</tool_response>" for o in outputs) + \
        "<|im_end|>\n"


def render_assistant(reasoning, content, calls):
    """A past assistant turn as the chat template renders it (used to check our verbatim appends)."""
    s = f"<|im_start|>assistant\n<think>\n{reasoning.strip()}\n</think>\n\n{content.strip()}"
    for i, (name, params) in enumerate(calls):
        s += ("\n\n" if content.strip() else "") if i == 0 else "\n"
        s += f"<tool_call>\n<function={name}>\n"
        s += "".join(f"<parameter={k}>\n{v}\n</parameter>\n" for k, v in params.items())
        s += "</function>\n</tool_call>"
    return s + "<|im_end|>\n"


def user_content(train, test_input, prompt):
    return build_prompt(train, test_input) + PROMPTS[prompt]


def initial_prompt(train, test_input, cfg):
    return render_system(cfg.tier, tool_specs(cfg)) + render_user(user_content(train, test_input, cfg.prompt)) \
        + GEN_PROMPT


# ---------------------------------------------------------------- parsing

_CALL = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)
_FUNC = re.compile(r"^\s*<function=([^>\n]+)>(.*?)</function>\s*$", re.DOTALL)
_PARAM = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)


def split_turn(text):
    """(reasoning, content) of one generated assistant turn; content is None while still thinking."""
    i = text.find(THINK_CLOSE)
    if i < 0:
        return text, None
    return text[:i], text[i + len(THINK_CLOSE):]


def parse_tool_calls(content):
    """Tool calls in the answer part of a turn: (calls, errors). calls = [(name, {param: str})].

    errors lists malformed blocks (unclosed <tool_call>, missing <function=...>)."""
    calls, errors = [], []
    for block in _CALL.findall(content):
        m = _FUNC.match(block)
        if not m:
            errors.append("malformed <tool_call> block: expected <function=NAME>...</function>")
            continue
        calls.append((m.group(1).strip(), {k.strip(): v for k, v in _PARAM.findall(m.group(2))}))
    tail = _CALL.sub("", content)
    if "<tool_call>" in tail or "<function=" in tail:
        errors.append("unclosed or malformed tool call: use <tool_call><function=NAME><parameter=code>..."
                      "</parameter></function></tool_call>")
    return calls, errors


def lenient_parse(text):
    """Last valid JSON grid in the text (trailing commas tolerated), else the last block of one-row-per-line."""
    text = text or ""
    for start in [i for i, ch in enumerate(text) if ch == "["][::-1]:
        if not re.match(r"\[\s*\[", text[start:start + 50]):
            continue
        depth = 0
        for j in range(start, len(text)):
            depth += (text[j] == "[") - (text[j] == "]")
            if depth == 0:
                block = text[start:j + 1]
                for candidate in (block, re.sub(r",\s*\]", "]", block)):
                    try:
                        grid = json.loads(candidate)
                    except json.JSONDecodeError:
                        continue
                    if valid_grid(grid):
                        return grid
                break
    rows, best = [], None
    for line in text.splitlines() + [""]:
        line = line.strip().rstrip(",")
        try:
            row = json.loads(line) if line.startswith("[") else None
        except json.JSONDecodeError:
            row = None
        if isinstance(row, list) and row and all(type(v) is int for v in row):
            rows.append(row)
            continue
        if rows and valid_grid(rows):
            best = rows
        rows = []
    return best


def budget_note(cfg, used):
    """Status line appended to the last tool output of a turn (E009 wording)."""
    if not cfg.budget_notes or used < cfg.budget_notes * cfg.max_total_output:
        return ""
    hard = cfg.max_total_output
    if cfg.budget_soft:
        note = (f"\n[budget: {used / 1000:.0f}k output tokens used; aim to finish within {cfg.budget_soft / 1000:.0f}k "
                f"(hard limit {hard / 1000:.0f}k)]")
        if cfg.budget_soft <= used < cfg.budget_warn * hard:
            note += (" Past the target: submit your best answer now with submit_answer, unless one more experiment "
                     "will very likely make every training pair pass." if cfg.soft_wording == "strong"
                     else " Past the target: submit as soon as a rule fits every training pair.")
    else:
        note = f"\n[budget: {used / 1000:.0f}k of {hard / 1000:.0f}k output tokens used]"
    if used >= cfg.budget_warn * hard:
        note += " Budget nearly spent: submit your best answer now with submit_answer (or at least set `answer`)."
    return note


# ---------------------------------------------------------------- model backend

@dataclass
class Completion:
    text: str
    completion_tokens: int
    prompt_tokens: int
    finish: str  # "stop" (EOS or stop string) or "length"


class OpenAICompletionsLLM:
    """Raw /v1/completions on an OpenAI-compatible server (SGLang, vLLM). Retries transient errors only:
    a completion has no side effects, so a retry is safe."""

    def __init__(self, base_url, api_key, model, temperature=1.0, top_p=0.95, top_k=20, timeout=3600,
                 max_retries=8, extra_body=None):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI(api_key=api_key or "none", base_url=base_url, timeout=timeout, max_retries=0)
        self.model, self.temperature, self.top_p, self.top_k = model, temperature, top_p, top_k
        self.max_retries, self.extra_body = max_retries, extra_body or {}
        self.transient_errors = 0

    async def complete(self, prompt, max_tokens, stop=None):
        import openai
        errs = 0
        while True:
            try:
                r = await self.client.completions.create(
                    model=self.model, prompt=prompt, max_tokens=max_tokens, temperature=self.temperature,
                    top_p=self.top_p, stop=stop, extra_body={"top_k": self.top_k, **self.extra_body})
                c = r.choices[0]
                return Completion(c.text, r.usage.completion_tokens, r.usage.prompt_tokens, c.finish_reason)
            except (openai.APIConnectionError, openai.APITimeoutError, openai.APIStatusError) as exc:
                status = getattr(exc, "status_code", None)
                if status is not None and status < 500 and status not in (408, 429):
                    raise
                errs += 1
                self.transient_errors += 1
                if errs > self.max_retries:
                    raise
                await asyncio.sleep(min(60, 5 * 2 ** min(errs - 1, 4)))

    async def close(self):
        await self.client.close()


# ---------------------------------------------------------------- agent loop

async def _call(fn, *args):
    # Blocking executor calls run in the loop's default thread pool. With many concurrent attempts, set a pool with
    # one thread per attempt: loop.set_default_executor(ThreadPoolExecutor(concurrency + 4)).
    return await asyncio.to_thread(fn, *args)


class ExecutorTimeout(RuntimeError):
    """An executor call did not return within its wall-clock cap; the session was replaced."""


def _quiet_close(session):
    try:
        session.close()
    except Exception:
        pass


class _Session:
    """The attempt's executor behind a hard wall-clock cap per call (the executor's own timeout is advisory: a
    wedged sandbox or child must not block the attempt). On a cap hit the session is abandoned (closed in a
    background thread) and a fresh one with only the task data takes its place."""

    def __init__(self, make_executor, train, test_input, call_limit, start_limit=180):
        self.make, self.train, self.test_input = make_executor, train, test_input
        self.call_limit, self.start_limit = call_limit, start_limit
        self.session, self.resets = None, 0

    async def start(self):
        session = self.make(self.train, self.test_input)
        try:
            await asyncio.wait_for(_call(session.__enter__), self.start_limit)
        except BaseException:
            threading.Thread(target=_quiet_close, args=(session,), daemon=True).start()
            raise
        self.session = session

    async def run(self, fn, *args):
        if self.session is None:
            raise RuntimeError("no executor session")
        try:
            return await asyncio.wait_for(_call(fn, self.session, *args), self.call_limit)
        except asyncio.TimeoutError:
            old, self.session = self.session, None
            threading.Thread(target=_quiet_close, args=(old,), daemon=True).start()
            self.resets += 1
            await self.start()
            raise ExecutorTimeout(f"executor call exceeded {self.call_limit:g}s") from None

    async def close(self):
        if self.session is not None:
            session, self.session = self.session, None
            try:
                await asyncio.wait_for(_call(session.close), 30)
            except BaseException:
                threading.Thread(target=_quiet_close, args=(session,), daemon=True).start()


def _execute(session, code):
    return session.execute(code)


def estimate_tokens(known_tokens, known_chars, text):
    """Prompt tokens of `text`, given that its first `known_chars` characters were measured as `known_tokens`.
    The unmeasured rest (tool output, the turn so far) is counted at 2 characters per token (conservative for
    JSON grids)."""
    return known_tokens + max(0, len(text) - known_chars + 1) // 2


async def solve(llm, make_executor, train, test_input, cfg):
    """One attempt. Returns (transcript, result). transcript is the full raw prompt+generation text.

    make_executor(train, test_input) -> executor (context manager with .execute(code)); train holds the
    training pairs with outputs (used host-side for --verify-submit), test_input the test grid only.
    Never raises (except on cancellation): errors are recorded, and any answer already set is still taken."""
    prompt = initial_prompt(train, test_input, cfg)
    used, turns, n_calls, n_tool_err, n_parse_err, n_eos, n_chunks = 0, 0, 0, 0, 0, 0, 0
    pred = source = stop = error = salvage_info = None
    max_prompt, content, submit_verified, forced_codes = 0, "", None, set()
    known_tokens, known_chars = 0, 0  # last measured prompt (tokens, chars) for the context estimate
    turn_log, t0 = [], time.monotonic()
    ex = _Session(make_executor, train, test_input, cfg.tool_timeout + cfg.exec_margin)

    async def complete(text, max_tokens, stop_strings=None):
        nonlocal used, n_chunks, max_prompt, known_tokens, known_chars
        c = await llm.complete(text, max_tokens, stop_strings) if stop_strings else await llm.complete(text, max_tokens)
        n_chunks += 1
        used += c.completion_tokens
        max_prompt = max(max_prompt, c.prompt_tokens)
        known_tokens, known_chars = c.prompt_tokens + c.completion_tokens, len(text) + len(c.text)
        return c

    async def submit(code):
        try:
            return await ex.run(submit_answer, code)
        except ExecutorTimeout:
            return "[answer not accepted] evaluation timed out; Python state lost (train/test_input/np restored)", None

    try:
        try:
            await ex.start()
            while True:
                if turns >= cfg.max_turns:
                    stop = "max_turns"
                    break
                turns += 1
                text, low_streak, after_eos, out_of_budget = "", 0, False, None
                turn_tokens = 0
                while True:  # one assistant turn, possibly over several requests
                    left = cfg.max_total_output - used
                    room = cfg.context - estimate_tokens(known_tokens, known_chars, prompt + text) - 64
                    if left <= 0 or room < 256:
                        out_of_budget = "output_budget" if left <= 0 else "context"
                        break
                    c = await complete(prompt + text, min(cfg.chunk_tokens, left, room))
                    turn_tokens += c.completion_tokens
                    text += c.text
                    if after_eos:
                        low_streak = low_streak + 1 if c.completion_tokens < LOW_YIELD else 0
                    after_eos = False
                    if c.finish == "length":
                        continue
                    if THINK_CLOSE not in text and n_eos < cfg.max_eos_continues and low_streak < 2:
                        n_eos += 1  # EOS inside thinking: let it carry on
                        after_eos = True
                        continue
                    break
                reasoning, answer_part = split_turn(text)
                entry = {"turn": turns, "tokens": turn_tokens, "closed_think": answer_part is not None}
                turn_log.append(entry)
                if out_of_budget or answer_part is None:
                    # Hard cap, context limit or thinking that never closes: force a submit_answer call; the
                    # session answer and salvage follow below.
                    stop = out_of_budget or "unclosed_think"
                    base = text + FORCE_THINK if answer_part is None else text[:len(text) - len(answer_part)] + "\n\n"
                    forced = base + FORCE_CALL
                    fits = (estimate_tokens(known_tokens, known_chars, prompt + forced) + cfg.force_submit_tokens
                            + 64 <= cfg.context)
                    if cfg.force_submit_tokens and fits:
                        c = await complete(prompt + forced, cfg.force_submit_tokens, ["</parameter>"])
                        expr = c.text.strip()
                        entry["forced_expr"] = expr[:200]
                        text = forced + c.text + "\n</parameter>\n</function>\n</tool_call>"
                        if expr:
                            out, grid = await submit(expr)
                            entry["forced_result"] = out[:200]
                            if grid is not None:
                                pred, source = grid, "forced"
                    prompt += text
                    break
                calls, perrs = parse_tool_calls(answer_part)
                if not calls and not perrs:  # a final text reply
                    content = answer_part
                    candidate = parse(answer_part)
                    if valid_grid(candidate):
                        pred, source = candidate, "parse"
                    else:
                        candidate = lenient_parse(answer_part)
                        if candidate is not None:
                            pred, source = candidate, "lenient"
                    prompt += text + "<|im_end|>\n"
                    stop = "final"
                    break
                outputs, accepted = [], None
                n_parse_err += len(perrs)
                for name, params in calls:
                    n_calls += 1
                    code = params.get("code")
                    if accepted is not None:
                        outputs.append("[skipped: answer already accepted]")
                        continue
                    if not isinstance(code, str) or name not in ("run_python", "submit_answer"):
                        n_parse_err += 1
                        n_tool_err += 1
                        outputs.append(f"[tool error: unknown tool {name!r} or missing `code` parameter; tools are "
                                       "run_python(code) and submit_answer(code)]")
                        continue
                    if name == "run_python":
                        try:
                            result = await ex.run(_execute, code)
                        except ExecutorTimeout as exc:
                            result = {"process_status": "timeout", "stdout": "", "state_reset": True,
                                      "stderr": f"[{exc}; the session was restarted]"}
                        n_tool_err += result["process_status"] != "completed"
                        outputs.append(tool_output(result, cfg.max_tool_output))
                        continue
                    out, grid = await submit(code)
                    if grid is not None and cfg.verify_submit:
                        try:
                            check = await ex.run(check_answer_on_train, code)
                        except ExecutorTimeout:
                            check = None
                        if check and check[0] and code not in forced_codes:
                            forced_codes.add(code)
                            out, grid = (f"[answer not accepted yet: the same expression with test_input = each "
                                         f"training input fails training pairs {check[0]} ({check[1]}). Fix the "
                                         "rule, or call submit_answer again with exactly the same code to submit "
                                         "anyway.]"), None
                        else:
                            submit_verified = None if check is None else not check[0]
                    n_tool_err += grid is None
                    outputs.append(out)
                    if grid is not None:
                        accepted = grid
                for e in perrs:
                    n_tool_err += 1
                    outputs.append(f"[tool error: {e}]")
                entry["calls"] = [name for name, _ in calls] + ["<malformed>"] * len(perrs)
                if accepted is not None:
                    pred, source, stop = accepted, "submit", "submitted"
                    prompt += text + "<|im_end|>\n"
                    break
                outputs[-1] += budget_note(cfg, used)
                prompt += text + "<|im_end|>\n" + render_tool_responses(outputs) + GEN_PROMPT
        except Exception as exc:
            error, stop = f"{type(exc).__name__}: {exc}"[:1000], "error"
        # Fallbacks, also after an error (R2): the session's `answer`, then the session's best function (R7).
        if pred is None and ex.session is not None:
            try:
                out, grid = await submit("answer")
                if grid is not None:
                    pred, source = grid, "session"
                elif cfg.salvage:
                    salvage_info, grid = await ex.run(salvage_answer)
                    if grid is not None:
                        pred, source = grid, "salvage"
            except Exception as exc:
                salvage_info = {"error": f"{type(exc).__name__}: {exc}"[:200]}
    finally:
        await ex.close()
    result = {"pred": pred, "answer_source": source if pred is not None else None, "content": content[-2000:],
              "stop_reason": stop, "turns": turns, "tool_calls": n_calls, "tool_errors": n_tool_err,
              "parse_failures": n_parse_err, "eos_continues": n_eos, "chunks": n_chunks,
              "usage": {"completion": used, "max_prompt": max_prompt}, "turn_log": turn_log,
              "executor_resets": ex.resets, "harness_version": HARNESS_VERSION,
              "seconds": round(time.monotonic() - t0, 1)}
    if salvage_info is not None:
        result["salvage"] = salvage_info
    if cfg.verify_submit:
        result["submit_verified"] = submit_verified if source == "submit" else None
        result["submit_rejections"] = len(forced_codes)
    if error:
        result["error"] = error
    return prompt, result


def config_dict(cfg, model):
    d = {**asdict(cfg), "model": model, "harness_version": HARNESS_VERSION}
    for key, default in (("soft_wording", "gentle"), ("context", CONTEXT), ("salvage", True), ("exec_margin", 35)):
        if d[key] == default:  # keep configs of runs made before the option existed unchanged
            del d[key]
    return d


The ARC Prize official prompt (exact bytes, no trailing newline, so it is written from Python rather than with `%%writefile`).

In [ ]:
ARC_PROMPT = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:"""
open('/kaggle/working/arc_solver/notes/prompts/arcprize_official_system_prompt.txt', 'w').write(ARC_PROMPT)

### Solver

Phases: fallback `submission.json` -> executor self-test -> offline install of vLLM from the attached wheel source (uv, pip fallback; versions logged) -> model preload -> vLLM server (retried once) -> scheduler with the tool agent -> summary. A watchdog restarts a dead server; an emergency timer guarantees a valid file.

In [ ]:
%%writefile /kaggle/working/arc_solver/solver.py
"""ARC Prize 2026 (ARC-AGI-2) offline solver: Qwen3.8-27B-FP8 on vLLM (TP=4, 4xL4), no tools.

One self-contained file; kaggle/build_notebook.py embeds it in the Kaggle notebook. Modes (env ARC_MODE):

- submit:   full run on the competition test file (used automatically on the competition rerun).
- fallback: write a valid fallback submission.json and exit (used for the commit run before submitting).
- bench:    test run: provenance hashes, server start, long-context decode benchmark, short end-to-end run on
            evaluation tasks with a near deadline, scoring against the evaluation solutions.
            ARC_HASH=0 / ARC_BENCH_DECODE=0 skip the first two (validation run: ARC_E2E_TASKS, ARC_E2E_MIN).

Robustness rules: a complete, valid submission.json is written before anything else and rewritten atomically
after every finished attempt; every failure path keeps the last valid file.

Prompt, parser and Qwen raw-template rendering follow evals/arc_harness/arcprize.py and evals/run_qwen_generate.py
(ARC Prize official prompt, tier xhigh, temp 1.0, top_p 0.95, top_k 20, forced final answer at the budget).
"""
import asyncio
import glob
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

T0 = float(os.environ.get("ARC_T0", time.time()))  # notebook start (set by the first cell)
MODE = os.environ.get("ARC_MODE", "submit")
WORK = "/kaggle/working"
WORK = os.environ.get("ARC_WORK", WORK)
OUT_SUB = os.path.join(WORK, "submission.json")
LOG_DIR = os.path.join(WORK, "arc_logs")
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
SERVED = os.environ.get("ARC_MODEL_ID", "qwen3.8-27b")
# Off-Kaggle validation: use an already running OpenAI-compatible server and skip install / launch.
EXTERNAL = os.environ.get("ARC_EXTERNAL_SERVER")  # e.g. http://localhost:30000/v1
if EXTERNAL:
    BASE = EXTERNAL.rstrip("/").removesuffix("/v1")
INPUT = os.environ.get("ARC_INPUT", "/kaggle/input")

# ---------------------------------------------------------------- settings (env overridable)

def envf(name, default):
    return type(default)(os.environ.get(name, default))

FORCE_AT_S = envf("ARC_FORCE_AT_S", 11.0 * 3600)    # force final answers for all in-flight attempts
HARD_STOP_S = envf("ARC_HARD_STOP_S", 11.5 * 3600)  # stop everything, final write (notebook guard: 11.75 h)
B_MAX1 = envf("ARC_B_MAX1", 96000)                  # attempt-1 budget cap (thinking + answer)
B_MIN1 = envf("ARC_B_MIN1", 16000)
B_MAX2 = envf("ARC_B_MAX2", 64000)
B_MIN2 = envf("ARC_B_MIN2", 32000)                  # attempt 2 is launched only if it can get this much
FINAL_TOKENS = envf("ARC_FINAL_TOKENS", 3000)
RATE_PRIOR = envf("ARC_RATE_PRIOR", 350.0)          # tok/s before the run measures its own rate
MAX_MODEL_LEN = envf("ARC_MAX_MODEL_LEN", 131072)
MAX_SEQS = envf("ARC_MAX_SEQS", 64)
CONCURRENCY = envf("ARC_CONCURRENCY", 0)            # 0 = from KV capacity
TOKENS_PER_SEQ_KV = envf("ARC_KV_PER_SEQ", 30000)   # KV tokens assumed per sequence when sizing concurrency
MTP = envf("ARC_MTP", 0)                            # speculative tokens (0 = off)
TIER = os.environ.get("ARC_TIER", "xhigh")
TEMPERATURE = envf("ARC_TEMPERATURE", 1.0)
GPU_UTIL = envf("ARC_GPU_UTIL", 0.95)
SERVER = os.environ.get("ARC_SERVER", "vllm")        # vllm (user's vllm-0.23.0 cu130 source) | sglang (ours)
AGENT = os.environ.get("ARC_AGENT", "tools")         # tools (E012 agent_qwen) | plain (v1 no-tool prompt)
TOOLS_CAP = envf("ARC_TOOLS_CAP", 48000)             # E012: hard cap per attempt (min with the adaptive share)
TOOLS_SOFT = envf("ARC_TOOLS_SOFT", 24000)
TOOLS_TIER = os.environ.get("ARC_TOOLS_TIER", "medium")
TOOL_TIMEOUT = envf("ARC_TOOL_TIMEOUT", 30.0)
LOCAL_FORCE_MIN = envf("ARC_LOCAL_FORCE_MIN", 40.0)  # local validation: force at max(T0 + this, ready + 20 min)
LOCAL_FORCE_CAP_MIN = envf("ARC_LOCAL_FORCE_CAP_MIN", 60.0)  # ... but never later than T0 + this
LOCAL_READY_MIN = envf("ARC_LOCAL_READY_MIN", 0.0)  # > 0: local force = server ready + this (overrides the above)
LOCAL_HARD_MIN = envf("ARC_LOCAL_HARD_MIN", 8.0)    # local hard stop = force + this
LOCAL_TASKS = os.environ.get("ARC_LOCAL_TASKS", "")  # comma-separated subset of the local test set
PUBLIC_TEST_IDS_SHA256 = "4cab35dea7bde6ee39bbfcfdcfc101153e19f333c14b68c68b84a7a3278c265e"  # public test file
PHASES = {}  # phase timings (s since T0), written to the summary
PKG = os.path.join(WORK, "arc_solver")  # evals/arc_harness + notes/prompts + solver.py, written by the notebook

PROMPT_TEMPLATE = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:"""  # noqa: F821  (ARC Prize official prompt, injected by build_notebook.py)

CLOSE = "</think>"
FORCE_TEXT = "\n\nI have to give my final answer now.\n</think>\n\n"
LOW_YIELD = 20
TIER_SYSTEM = {
    "xhigh": "Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, "
             "consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final "
             "answer.",
    "medium": None,
}


def log(*a):
    el = time.time() - T0
    print(f"[{el / 3600:5.2f}h {time.strftime('%H:%M:%S')}]", *a, flush=True)


# ---------------------------------------------------------------- prompt / parse (verbatim from arc_harness)

def build_prompt(train, test_input):
    ex = ""
    for i, pair in enumerate(train):
        ex += f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(pair["input"]) + "\n\n"
        ex += "OUTPUT: \n\n" + json.dumps(pair["output"]) + "\n\n"
    return PROMPT_TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))


def render_prompt(user_content, tier=TIER):
    s = ""
    if TIER_SYSTEM.get(tier):
        s += f"<|im_start|>system\n{TIER_SYSTEM[tier]}<|im_end|>\n"
    return s + f"<|im_start|>user\n{user_content.strip()}<|im_end|>\n<|im_start|>assistant\n<think>\n"


def answer_section(gen):
    i = gen.rfind(CLOSE)
    return None if i < 0 else gen[i + len(CLOSE):]


def is_grid(x):
    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)


def extract_from_boxed(s):
    m = re.search(r"\\boxed\{(.*?)\}", s, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            return g if is_grid(g) else None
        except json.JSONDecodeError:
            return None
    return None


def backscan_json_parser(s):
    end = max(s.rfind("]"), s.rfind("}"))
    if end < 0:
        return None
    close = s[end]
    open_ = "[" if close == "]" else "{"
    depth = 1
    for i in range(end - 1, -1, -1):
        depth += (s[i] == close) - (s[i] == open_)
        if depth == 0:
            try:
                g = json.loads(s[i:end + 1])
                return g if is_grid(g) else None
            except json.JSONDecodeError:
                return None
    return None


def parse(s):
    return extract_from_boxed(s or "") or backscan_json_parser(s or "")


def valid_grid(grid):
    return (isinstance(grid, list) and 1 <= len(grid) <= 30
            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)
            and len({len(row) for row in grid}) == 1
            and all(type(v) is int and 0 <= v <= 9 for row in grid for v in row))


# ---------------------------------------------------------------- paths

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None


def comp_file(name):
    return (find_one(f"{INPUT}/competitions/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/**/{name}"))


def model_dir():
    for pat in [f"{INPUT}/models/mikedan7/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/**/qwen3-8-27b-fp8*/**/config.json"]:
        p = find_one(pat)
        if p:
            return os.path.dirname(p)
    raise FileNotFoundError("model dir")


def wheel_dir():
    """Our own wheel sources (attached as kernel sources): SGLang 0.5.20 cu130 or the user's vLLM 0.23.0 cu128."""
    pat = "sglang-0.5.20-*.whl" if SERVER == "sglang" else "vllm-0.23.0*.whl"
    p = find_one(f"{INPUT}/notebooks/chankhavu/**/{pat}") or find_one(f"{INPUT}/**/{pat}")
    if not p:
        raise FileNotFoundError(f"wheel source for {SERVER}")
    return os.path.dirname(p)


# ---------------------------------------------------------------- submission

class Submission:
    def __init__(self, tasks):
        self.tasks = tasks
        self.answers = {}  # (tid, pair) -> {attempt_index: grid}

    def fallback(self, tid, p):
        g = self.tasks[tid]["test"][p]["input"]
        return g if valid_grid(g) else [[0]]

    def build(self):
        sub = {}
        for tid, task in self.tasks.items():
            lst = []
            for p in range(len(task["test"])):
                got = [g for _, g in sorted(self.answers.get((tid, p), {}).items()) if valid_grid(g)]
                a1 = got[0] if got else self.fallback(tid, p)
                a2 = next((g for g in got[1:] if g != a1), None)
                if a2 is None:
                    a2 = self.fallback(tid, p) if a1 != self.fallback(tid, p) else a1
                lst.append({"attempt_1": a1, "attempt_2": a2})
            sub[tid] = lst
        return sub

    def write(self):
        sub = self.build()
        tmp = OUT_SUB + ".tmp"
        with open(tmp, "w") as f:
            json.dump(sub, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, OUT_SUB)


def check_submission(path, tasks):
    sub = json.load(open(path))
    assert set(sub) == set(tasks), "task ids differ"
    for tid, task in tasks.items():
        assert len(sub[tid]) == len(task["test"]), tid
        for slot in sub[tid]:
            assert valid_grid(slot["attempt_1"]) and valid_grid(slot["attempt_2"]), tid
    return True


# ---------------------------------------------------------------- environment

def sh(cmd, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or "") + (r.stderr or "")


def driver_env():
    """Host driver libcuda first (images also carry an older compat libcuda); set before any CUDA process starts."""
    real = "/usr/local/nvidia/lib64"
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"


def install_server():
    wd = wheel_dir()
    t = time.time()
    if SERVER == "sglang":
        cmd = f"{sys.executable} -m pip install --no-index --find-links {wd} -r {wd}/requirements.lock"
    else:
        cmd = (f"{sys.executable} -m pip install --no-index --find-links {wd} vllm==0.23.0 flashinfer-python "
               f"flashinfer-jit-cache")
    code, out = sh("uv " + cmd.replace(f"{sys.executable} -m pip install", "pip install --system --python "
                                       + sys.executable), timeout=2400)
    log(f"uv offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-1500:])
        t = time.time()
        code, out = sh(cmd, timeout=2400)
        log(f"pip offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-4000:])
        raise RuntimeError("pip install failed")
    mod = "sglang" if SERVER == "sglang" else "vllm"
    code, out = sh(f"{sys.executable} -c 'import torch; print(torch.version.cuda)'", timeout=600)
    if out.strip().splitlines()[-1:] == ["None"]:
        # A CPU torch of the same version in the image satisfies the pin and is kept: force the CUDA build.
        tw = sorted(glob.glob(f"{wd}/torch-*.whl")) + sorted(glob.glob(f"{wd}/torchvision-*.whl")) + \
            sorted(glob.glob(f"{wd}/torchaudio-*.whl"))
        code, out2 = sh(f"{sys.executable} -m pip install --no-index --force-reinstall --no-deps " + " ".join(tw),
                        timeout=1800)
        log(f"CPU torch found; reinstalled CUDA torch from the wheel source: rc={code}")
    code, out = sh(f"{sys.executable} -c 'import torch, {mod}; print(torch.__version__, torch.version.cuda, "
                   f"torch.cuda.is_available(), torch.cuda.device_count(), {mod}.__version__)'", timeout=600)
    log("torch/server:", out.strip().splitlines()[-1] if out.strip() else out)
    if SERVER == "vllm":
        _, ldd = sh(f"ldd $({sys.executable} -c 'import glob, os, vllm; "
                    f"print(glob.glob(os.path.dirname(vllm.__file__) + \"/_C*.so\")[0])') | grep -i 'not found'",
                    timeout=120)
        log("vllm._C missing libraries:", ldd.strip() or "none")
    _, mem = sh("free -g | head -2", timeout=30)
    log("host memory:", " | ".join(mem.strip().splitlines()))
    _, freeze = sh(f"{sys.executable} -m pip freeze", timeout=300)
    open(os.path.join(LOG_DIR, "pip_freeze.txt"), "w").write(freeze)
    log("installed:", " ".join(l for l in freeze.splitlines() if l.split("==")[0].lower() in
                               ("vllm", "torch", "transformers", "flashinfer-python", "triton", "sglang", "numpy")))


def file_hashes(paths, workers=16):
    def h(p):
        s256, s1 = hashlib.sha256(), hashlib.sha1()
        size = os.path.getsize(p)
        s1.update(f"blob {size}\0".encode())
        with open(p, "rb") as f:
            while True:
                b = f.read(1 << 24)
                if not b:
                    break
                s256.update(b)
                s1.update(b)
        return p, {"size": size, "sha256": s256.hexdigest(), "git_sha1": s1.hexdigest()}
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(h, paths))


def preload(mdir, workers=16):
    files = [os.path.join(r, f) for r, _, fs in os.walk(mdir) for f in fs]

    def rd(p):
        with open(p, "rb") as f:
            while f.read(1 << 26):
                pass
    t = time.time()
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(rd, files))
    log(f"preloaded {len(files)} model files in {time.time() - t:.0f}s")


def launch_server(mdir):
    env = os.environ.copy()
    env.update(PYTORCH_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1",
               TRANSFORMERS_OFFLINE="1", OMP_NUM_THREADS="1")
    if SERVER == "sglang":
        cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", mdir, "--served-model-name", SERVED,
               "--host", "127.0.0.1", "--port", str(PORT), "--tp-size", "4", "--kv-cache-dtype", "fp8_e4m3",
               "--context-length", str(MAX_MODEL_LEN), "--mem-fraction-static", os.environ.get("ARC_SGL_MEM", "0.85"),
               "--max-running-requests", str(MAX_SEQS), "--trust-remote-code",
               "--attention-backend", os.environ.get("ARC_SGL_ATTN", "triton"), "--log-level", "info"]
        if os.environ.get("ARC_RADIX", "1") != "1":
            cmd.append("--disable-radix-cache")
    else:
        attn = "TRITON_ATTN"  # FlashInfer fp8-KV kernels need SM90+; L4 is SM89
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", mdir,
               "--served-model-name", SERVED, "--host", "127.0.0.1", "--port", str(PORT),
               "--tensor-parallel-size", "4", "--gpu-memory-utilization", str(GPU_UTIL),
               "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_SEQS),
               "--kv-cache-dtype", "fp8", "--attention-backend", attn, "--enable-prefix-caching",
               "--trust-remote-code", "--no-async-scheduling"]
        if MTP > 0:
            cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": MTP})]
        env.update(VLLM_FORCE_ATTN_BACKEND=attn, VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1")
    log("server cmd:", " ".join(cmd))
    path = os.path.join(LOG_DIR, f"{SERVER}_server.log")
    fh = open(path, "a")
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            start_new_session=True, env=env)
    return proc, path


def kill_server(proc=None):
    """Kill the server's whole process tree (vLLM renames its workers VLLM::Worker_TP*), then wait for GPU memory."""
    if proc is not None:
        try:
            os.killpg(proc.pid, 9)
        except Exception:
            pass
    sh("pkill -9 -f '[v]llm.entrypoints' ; pkill -9 -f '[V]LLM::' ; pkill -9 -f '[s]glang.launch_server' ; "
       "pkill -9 -f '[s]glang::' ; sleep 5", timeout=60)
    for _ in range(24):
        _, out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", timeout=30)
        used = [int(x) for x in out.split() if x.strip().isdigit()]
        if used and max(used) < 2000:
            break
        time.sleep(5)


def server_log_tail(path, n=60):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception:
        return ""


def wait_ready(proc, path, timeout):
    import urllib.request
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            log(f"server exited rc={proc.returncode}\n" + server_log_tail(path))
            return False
        try:
            with urllib.request.urlopen(BASE + "/v1/models", timeout=5) as r:
                if r.status == 200:
                    log(f"server ready in {time.time() - t:.0f}s")
                    return True
        except Exception:
            pass
        time.sleep(5)
    log("server not ready in time\n" + server_log_tail(path))
    return False


def server_capacity(path):
    """(kv_tokens, max_running) from the server log."""
    txt = open(path, errors="ignore").read()
    if SERVER == "sglang":
        kv = re.findall(r"max_total_num_tokens\W{1,3}(\d+)", txt)
        mr = re.findall(r"max_running_requests\W{1,3}(\d+)", txt)
        out = (int(kv[-1]) if kv else None, int(mr[-1]) if mr else None)
    else:
        m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", txt)
        out = (int(m[-1].replace(",", "")) if m else None, None)
    log("server capacity (kv tokens, max running):", out)
    return out


# ---------------------------------------------------------------- client

class Client:
    def __init__(self):
        import httpx
        self.last_prompt_tokens = None
        self.http = httpx.AsyncClient(base_url=BASE, timeout=httpx.Timeout(connect=30.0, read=900.0, write=120.0, pool=None),
                                      limits=httpx.Limits(max_connections=512, max_keepalive_connections=512))

    async def count(self, prompt):
        try:
            r = await self.http.post("/tokenize", json={"model": SERVED, "prompt": prompt})
            return int(r.json()["count"])
        except Exception:
            return int(len(prompt) / 2.5)

    async def stream(self, prompt, max_tokens, deadline, on_tokens, stop=None, info=None):
        """Streamed completion bounded by `deadline` even when no data arrives (finish 'deadline')."""
        parts = {"text": [], "ntok": 0}
        try:
            return await asyncio.wait_for(self._stream(prompt, max_tokens, deadline, on_tokens, stop, info, parts),
                                          timeout=max(1.0, deadline - time.time() + 5))
        except asyncio.TimeoutError:
            return "".join(parts["text"]), parts["ntok"], "deadline"

    async def _stream(self, prompt, max_tokens, deadline, on_tokens, stop, info, parts):
        """Streamed completion. Returns (text, completion_tokens, finish_reason); finish 'deadline' if cut off.

        Breaking out of the stream closes the connection, which makes vLLM abort the request.
        """
        body = {"model": SERVED, "prompt": prompt, "max_tokens": max_tokens, "temperature": TEMPERATURE,
                "top_p": 0.95, "top_k": 20, "stream": True, "skip_special_tokens": False,
                "stream_options": {"include_usage": True, "continuous_usage_stats": True}}
        if stop:
            body["stop"] = stop
        text, ntok, finish = parts["text"], 0, None
        async with self.http.stream("POST", "/v1/completions", json=body) as r:
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {(await r.aread())[:300]!r}")
            async for line in r.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[6:]
                if data.strip() == "[DONE]":
                    break
                d = json.loads(data)
                if d.get("error"):
                    raise RuntimeError(f"server error event: {str(d['error'])[:300]}")
                u = d.get("usage")
                got_text = False
                for c in d.get("choices") or []:
                    text.append(c.get("text") or "")
                    got_text = got_text or bool(c.get("text"))
                    if c.get("finish_reason"):
                        finish = c["finish_reason"]
                if u and u.get("completion_tokens") is not None:
                    self.last_prompt_tokens = u.get("prompt_tokens") or self.last_prompt_tokens
                    if info is not None and u.get("prompt_tokens"):
                        info["prompt_tokens"] = u["prompt_tokens"]
                    on_tokens(u["completion_tokens"] - ntok)
                    ntok = u["completion_tokens"]
                elif got_text:  # server without continuous usage stats: ~1 token per chunk until usage arrives
                    on_tokens(1)
                    ntok += 1
                parts["ntok"] = ntok
                if time.time() > deadline:
                    finish = "deadline"
                    break
        return "".join(text), ntok, finish or "stop"


# ---------------------------------------------------------------- tool executor (Kaggle hardening)

def robust_executor_factory(timeout):
    """E012 LocalExecutor (qwen-tools-3 executor: RLIMIT_AS/CPU/FSIZE, nice, oom_score_adj=1000, SIGINT to the
    process group, retried start with a longer timeout). nice 15 so tool code never starves the model server."""
    from arc_harness import executor as EX
    try:
        return EX.local_factory(timeout=timeout, nice=15)
    except TypeError:  # older executor without the hardening knobs
        return EX.local_factory(timeout=timeout)


# ---------------------------------------------------------------- scheduler

class Scheduler:
    def __init__(self, tasks, sub, client, force_at, hard_stop, concurrency, attempts=2, tag="run"):
        self.tasks, self.sub, self.client = tasks, sub, client
        self.force_at, self.hard_stop = force_at, hard_stop
        self.concurrency, self.attempts = concurrency, attempts
        self.pending1 = []
        self.pending2 = []
        self.inflight = {}  # job -> [budget, used]
        self.tokens = 0
        self.rate_samples = []  # (time, total tokens)
        self.gen_start = None
        self.records = []
        self.rec_path = os.path.join(LOG_DIR, f"attempts_{tag}.jsonl")
        self.launch_margin = 120  # no new attempts in the last 2 min before forcing
        self.retries = {}
        self.server_down = lambda: False  # set by main(): True while the vLLM server is dead or restarting

    def on_tokens(self, n):
        self.tokens += n
        if "first_token" not in PHASES and n > 0:
            PHASES["first_token"] = round(time.time() - T0)

    def rate(self):
        now = time.time()
        self.rate_samples.append((now, self.tokens))
        self.rate_samples = [s for s in self.rate_samples if now - s[0] < 1800] or [(now, self.tokens)]
        t_old, k_old = self.rate_samples[0]
        if self.gen_start is None or now - self.gen_start < 900 or now - t_old < 300:
            return RATE_PRIOR
        return max(50.0, (self.tokens - k_old) / (now - t_old))

    def capacity_left(self):
        cap = self.rate() * max(0.0, self.force_at - time.time())
        return cap - sum(max(0, b - u) for b, u in self.inflight.values())

    def next_job(self):
        """(job, budget) or None. Attempt-1 jobs first; attempt 2 only with spare capacity."""
        if self.pending1:
            job = self.pending1.pop(0)
            share = self.capacity_left() / (len(self.pending1) + 1)
            return job, int(min(B_MAX1, max(B_MIN1, share)))
        while self.pending2:
            job = self.pending2.pop(0)
            share = self.capacity_left() / (len(self.pending2) + 1)
            if share < B_MIN2:
                continue
            return job, int(min(B_MAX2, share))
        return None

    async def attempt(self, job, budget):
        if AGENT == "tools":
            return await self.attempt_tools(job, budget)
        return await self.attempt_plain(job, budget)

    async def attempt_tools(self, job, budget):
        """One E012 tool-agent attempt (agent_qwen.solve + LocalExecutor) with deadline forcing.

        DeadlineLLM streams each completion; past force_at it cuts the stream and reports the attempt's budget as
        used, so the agent goes straight to its forced submit_answer (then the session `answer`). The forced call
        (stop=["</parameter>"]) may run until the hard stop."""
        import dataclasses
        from arc_harness import agent_qwen as AQ
        tid, p, a = job
        task = self.tasks[tid]
        cap = int(min(TOOLS_CAP, budget))
        extra = {"context": MAX_MODEL_LEN} if "context" in AQ.AgentConfig.__dataclass_fields__ else {}
        cfg = AQ.AgentConfig(**extra, tier=TOOLS_TIER, max_total_output=cap, budget_soft=min(TOOLS_SOFT, cap),
                             budget_notes=0.01, budget_warn=0.75, verify_submit=False, max_turns=60,
                             chunk_tokens=16384, force_submit_tokens=256, tool_timeout=TOOL_TIMEOUT,
                             temperature=TEMPERATURE, top_p=0.95, top_k=20)
        sch, client = self, self.client
        stats = {"tokens": 0, "requests": 0, "deadline_cuts": 0, "stream_errors": 0}

        class DeadlineLLM:
            async def complete(self, prompt, max_tokens, stop=None):
                info = {}
                stats["requests"] += 1
                if stop is None and time.time() >= sch.force_at:
                    stats["deadline_cuts"] += 1
                    return AQ.Completion("", cap, 0, "length")
                # forced submit expression: must end well before the hard-stop cancel so that submit_answer and
                # the session-answer fallback still run (review S2)
                dl = (max(sch.hard_stop - 180, min(time.time() + 30, sch.hard_stop - 10)) if stop
                      else sch.force_at)

                def cb(n):
                    stats["tokens"] += n
                    sch.on_tokens(n)
                    if job in sch.inflight:
                        sch.inflight[job][1] = stats["tokens"]
                try:
                    text, ntok, finish = await client.stream(prompt, max_tokens, dl, cb, stop=stop, info=info)
                except Exception as exc:
                    stats["stream_errors"] += 1
                    stats["last_stream_error"] = f"{type(exc).__name__}: {exc}"[:300]
                    if sch.server_down():
                        raise  # attempt is requeued once the server is back
                    # otherwise: end the attempt through the forced submit / session answer path
                    return AQ.Completion("", cap, 0, "length")
                if finish == "deadline":
                    stats["deadline_cuts"] += 1
                    return AQ.Completion(text, ntok + cap, info.get("prompt_tokens", 0), "length")
                return AQ.Completion(text, ntok, info.get("prompt_tokens", 0),
                                     "length" if finish == "length" else "stop")

        self.inflight[job] = [cap, 0]
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": cap, "agent": "tools"}
        t0 = time.time()
        try:
            transcript, res = await AQ.solve(DeadlineLLM(), robust_executor_factory(TOOL_TIMEOUT), task["train"],
                                             task["test"][p]["input"], cfg)
        except Exception as exc:
            transcript, res = "", {"pred": None, "error": f"{type(exc).__name__}: {exc}"[:400]}
        finally:
            self.inflight.pop(job, None)
        pred = res.get("pred")
        if hasattr(pred, "tolist"):
            pred = pred.tolist()
        rec.update(tokens=stats["tokens"], requests=stats["requests"], deadline_cuts=stats["deadline_cuts"],
                   stream_errors=stats["stream_errors"], last_stream_error=stats.get("last_stream_error"),
                   answered=pred is not None and valid_grid(pred), pred=pred,
                   forced=res.get("answer_source") in ("forced", "session") or stats["deadline_cuts"] > 0,
                   force_reason=("deadline" if stats["deadline_cuts"] else res.get("stop_reason")),
                   elapsed_s=round(time.time() - t0),
                   **{k: res.get(k) for k in ("answer_source", "stop_reason", "turns", "tool_calls", "tool_errors",
                                              "parse_failures", "eos_continues", "chunks", "seconds", "turn_log")})
        if res.get("error"):
            rec["error"] = res["error"]
        try:
            with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
                f.write(transcript)
        except Exception:
            pass
        return rec

    async def attempt_plain(self, job, budget):
        tid, p, a = job
        task = self.tasks[tid]
        raw = render_prompt(build_prompt(task["train"], task["test"][p]["input"]))
        ptok = await self.client.count(raw)
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": budget, "prompt_tokens": ptok}
        self.inflight[job] = [budget, 0]
        gen, used, conts, streak, forced, force_reason, after_eos, chunks = "", 0, 0, 0, False, None, False, []
        t0 = time.time()
        try:
            while True:
                room = MAX_MODEL_LEN - ptok - used - 64
                if forced:
                    mt, dl = min(FINAL_TOKENS, room), self.hard_stop
                else:
                    mt, dl = min(budget - used, room - FINAL_TOKENS), self.force_at
                if mt <= 0 and not forced:
                    forced, force_reason = True, "budget"
                    gen += FORCE_TEXT
                    continue
                if mt <= 0:
                    break
                text, ntok, finish = await self.client.stream(raw + gen, mt, dl, self.on_tokens)
                gen += text
                used += ntok
                self.inflight[job][1] = used
                chunks.append([ntok, finish])
                if after_eos:
                    streak = streak + 1 if ntok < LOW_YIELD else 0
                if forced:
                    break
                if answer_section(gen) is not None:
                    if finish == "length" and used < budget and room - ntok > 64:
                        continue
                    break
                if finish == "deadline" or used >= budget or finish == "length":
                    forced, force_reason = True, ("deadline" if finish == "deadline" else "budget")
                    gen += FORCE_TEXT
                    continue
                # EOS mid-thought: continue, force after two low-yield continuations in a row
                if streak >= 2:
                    forced, force_reason = True, "low_yield_eos"
                    gen += FORCE_TEXT
                    continue
                conts += 1
                after_eos = True
        except Exception as exc:
            rec["error"] = f"{type(exc).__name__}: {exc}"[:400]
        finally:
            self.inflight.pop(job, None)
        content = answer_section(gen)
        pred = parse(content) if content is not None else None
        rec.update(tokens=used, continuations=conts, forced=forced, force_reason=force_reason, chunks=chunks,
                   elapsed_s=round(time.time() - t0), answered=pred is not None and valid_grid(pred), pred=pred,
                   tail=(content or gen[-300:])[-300:])
        with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
            f.write(gen)
        return rec

    async def worker(self, wid):
        while time.time() < self.force_at - self.launch_margin:
            nj = self.next_job()
            if nj is None:
                return
            job, budget = nj
            rec = await self.attempt(job, budget)
            if "error" in rec and not rec["answered"] and self.server_down():
                # server crashed: requeue the job (at most twice) and wait for the watchdog to restart it
                self.retries[job] = self.retries.get(job, 0) + 1
                if self.retries[job] <= 2:
                    (self.pending1 if job[2] == 0 else self.pending2).insert(0, job)
                    log(f"server down; requeued {job}, waiting")
                    while self.server_down() and time.time() < self.force_at - self.launch_margin:
                        await asyncio.sleep(15)
                    continue
            self.records.append(rec)
            with open(self.rec_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
            tid, p, a = job
            if rec["answered"]:
                self.sub.answers.setdefault((tid, p), {})[a] = rec["pred"]
                self.sub.write()
            if a == 0 and self.attempts > 1:
                self.pending2.append((tid, p, 1))
            log(f"done {tid}.{p}#{a} answered={rec['answered']} tok={rec['tokens']} budget={budget} "
                f"forced={rec['forced']}/{rec['force_reason']} "
                f"{rec['elapsed_s']}s {rec.get('error', '')}")

    async def monitor(self):
        last_t, last_k = time.time(), 0
        while True:
            await asyncio.sleep(120)
            now = time.time()
            r = (self.tokens - last_k) / (now - last_t)
            last_t, last_k = now, self.tokens
            log(f"progress: {len(self.records)} attempts done, in flight {len(self.inflight)}, pending1 "
                f"{len(self.pending1)}, pending2 {len(self.pending2)}, tokens {self.tokens}, window {r:.0f} tok/s, "
                f"rate est {self.rate():.0f} tok/s, avg {self.tokens / max(1, now - self.gen_start):.0f} tok/s")

    async def run(self, jobs1):
        self.pending1 = list(jobs1)
        self.gen_start = time.time()
        mon = asyncio.create_task(self.monitor())
        try:
            await asyncio.wait_for(asyncio.gather(*(self.worker(i) for i in range(self.concurrency))),
                                   timeout=max(60, self.hard_stop - time.time()))
        except asyncio.TimeoutError:
            log("hard stop reached; cancelling in-flight attempts")
        mon.cancel()
        el = time.time() - self.gen_start
        log(f"scheduler finished: {len(self.records)} attempts, {self.tokens} tokens in {el / 60:.1f} min "
            f"({self.tokens / max(1, el):.0f} tok/s)")


def order_jobs(tasks):
    """Attempt-1 jobs, shortest prompts first (approximated by character length)."""
    jobs = [(tid, p, 0) for tid, t in tasks.items() for p in range(len(t["test"]))]
    size = {tid: len(json.dumps(t["train"])) + len(json.dumps(t["test"])) for tid, t in tasks.items()}
    return sorted(jobs, key=lambda j: (size[j[0]], j[0], j[1]))


# ---------------------------------------------------------------- benchmark (test runs only)

async def bench_decode(client, prefixes, n_seqs, gen_tokens, tasks, max_chars=None, n_unique=None):
    """Decode throughput at long context: real ARC prompts + real Qwen trace prefixes, n_seqs in parallel.

    n_unique: number of distinct prompts (sequence i uses prompt i % n_unique), so repeats can hit a prefix cache.
    """
    total = {"tok": 0}
    prompts = []
    for i in range(n_seqs):
        pr = prefixes[i % (n_unique or len(prefixes))]
        t = tasks[pr["task_id"]]
        prompts.append(render_prompt(build_prompt(t["train"], t["test"][pr["pair"]]["input"]))
                       + pr["text"][:max_chars])
    ctx = [await client.count(p) for p in prompts]
    log(f"bench: {n_seqs} seqs, context tokens mean {sum(ctx) / len(ctx):.0f} min {min(ctx)} max {max(ctx)}")
    first = {}
    t_start = time.time()

    def cb(n):
        total["tok"] += n

    async def one(i):
        def cbi(n):
            if i not in first:
                first[i] = time.time()
            cb(n)
        return await client.stream(prompts[i], gen_tokens, time.time() + 3600, cbi)

    samples = []

    async def sampler():
        while True:
            await asyncio.sleep(10)
            samples.append((time.time(), total["tok"]))
    s = asyncio.create_task(sampler())
    res = await asyncio.gather(*(one(i) for i in range(n_seqs)), return_exceptions=True)
    s.cancel()
    t_end = time.time()
    t_all_first = max(first.values()) if first else t_end
    dec = [x for x in samples if x[0] >= t_all_first]
    steady = (dec[-1][1] - dec[0][1]) / (dec[-1][0] - dec[0][0]) if len(dec) >= 2 else None
    errs = [repr(r)[:200] for r in res if isinstance(r, Exception)]
    out = {"n_seqs": n_seqs, "ctx_mean": sum(ctx) / len(ctx), "gen_tokens": gen_tokens, "total_tokens": total["tok"],
           "wall_s": t_end - t_start, "prefill_phase_s": t_all_first - t_start, "steady_decode_tok_s": steady,
           "overall_tok_s": total["tok"] / (t_end - t_start), "errors": errs}
    log("bench result:", json.dumps(out))
    return out


def score(sub, solutions, tasks):
    tot, pairs = 0.0, 0
    for tid, t in tasks.items():
        n = len(t["test"])
        ok = 0
        for p in range(n):
            s = sub[tid][p]
            ok += solutions[tid][p] in (s["attempt_1"], s["attempt_2"])
        tot += ok / n
        pairs += n
    return tot, len(tasks)


# ---------------------------------------------------------------- main

def setup_harness():
    """Make the bundled E012 harness importable and self-test the executor; False -> use the plain agent."""
    try:
        sys.path.insert(0, os.path.join(PKG, "evals"))
        from arc_harness import agent_qwen
        from arc_harness.executor import LocalExecutor, submit_answer
        agent_qwen.CONTEXT = MAX_MODEL_LEN  # the server's context, not the model's 262k
        t = time.time()
        with LocalExecutor([{"input": [[1, 2]], "output": [[2, 1]]}], [[3, 4]], timeout=3) as ex:
            r1 = ex.execute("import numpy as np\nx = np.array(test_input)[:, ::-1]\nprint(x.tolist())")
            r2 = ex.execute("print(x.sum())")
            r3 = ex.execute("1/0")
            r4 = ex.execute("while True:\n    pass")
            r5 = ex.execute("print(x.shape)")
            out, grid = submit_answer(ex, "x")
        ok = (r1["stdout"].strip() == "[[4, 3]]" and r2["stdout"].strip() == "7" and "ZeroDivisionError" in r3["stderr"]
              and r4["process_status"] == "timeout" and r5["stdout"].strip() == "(1, 2)" and grid == [[4, 3]])
        log(f"harness {agent_qwen.HARNESS_VERSION}; executor self-test {'OK' if ok else 'FAILED'} in "
            f"{time.time() - t:.1f}s")
        if not ok:
            log("self-test results:", json.dumps([r1, r2, r3, r4, r5, out])[:1500])
        return ok
    except Exception as exc:
        log(f"harness/executor self-test error: {type(exc).__name__}: {exc}")
        return False


def is_rerun(test_path):
    """Belt and braces: the env var, or a test file whose task ids differ from the public one."""
    import hashlib
    env = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
    try:
        ids = sorted(json.load(open(test_path)))
        differs = hashlib.sha256(",".join(ids).encode()).hexdigest() != PUBLIC_TEST_IDS_SHA256
    except Exception:
        differs = False
    return env or differs, env, differs


def emergency_timer():
    """Last-resort timer (review S1): at T0 + ARC_EMERGENCY_S, make sure a valid submission.json exists, kill the
    server and all children, and exit, whatever the asyncio side is doing."""
    import threading

    def fire():
        log("EMERGENCY timer fired: validating submission.json, killing children, exiting")
        try:
            path = os.path.join(WORK, "submission.json")
            tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
            try:
                check_submission(path, tasks)
            except Exception:
                global OUT_SUB
                OUT_SUB = path
                Submission(tasks).write()
        finally:
            subprocess.run("pkill -9 -f '[V]LLM::'; pkill -9 -f '[v]llm.entrypoints'; pkill -9 -f '[s]glang'",
                           shell=True)
            os._exit(0)
    t = threading.Timer(max(1.0, T0 + envf("ARC_EMERGENCY_S", 11.7 * 3600) - time.time()), fire)
    t.daemon = True
    t.start()


def main():
    global OUT_SUB
    emergency_timer()
    os.makedirs(LOG_DIR, exist_ok=True)
    driver_env()
    global AGENT
    rerun, env_rerun, ids_differ = is_rerun(comp_file("arc-agi_test_challenges.json"))
    mode = "submit" if rerun else MODE
    log(f"rerun={rerun} (env {env_rerun}, test ids differ from public {ids_differ})")
    log(f"mode={mode} server={SERVER} agent={AGENT} rerun={rerun} "
        f"T0={time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(T0))} UTC")
    test_tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
    Submission(test_tasks).write()  # valid fallback for the competition test file, always
    log(f"fallback submission written ({len(test_tasks)} tasks)")
    if mode == "fallback":
        return
    if mode == "local":
        # Non-rerun commit/interactive run: small local test set with known solutions, self-scored, time-boxed.
        tasks = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_challenges.json")))
        solutions = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_solutions.json")))
        if LOCAL_TASKS:
            tasks = {t: tasks[t] for t in LOCAL_TASKS.split(",") if t in tasks}
        force_at = hard_stop = None  # set once the server is ready
        OUT_SUB = os.path.join(WORK, "local_submission.json")
    else:
        tasks, solutions = test_tasks, None
        force_at, hard_stop = T0 + FORCE_AT_S, T0 + HARD_STOP_S
    sub = Submission(tasks)
    sub.write()
    log(f"{mode}: {len(tasks)} tasks, {sum(len(t['test']) for t in tasks.values())} test inputs"
        + (f"; force at {(force_at - T0) / 3600:.2f}h, hard stop {(hard_stop - T0) / 3600:.2f}h" if force_at else ""))
    if AGENT == "tools" and not setup_harness():
        AGENT = "plain"
        log("falling back to the plain (no-tool) agent")
    PHASES["harness_ready"] = round(time.time() - T0)

    if EXTERNAL:
        # off-Kaggle validation against an already running server: no install, no launch, no watchdog restarts
        class _Alive:
            returncode = None

            def poll(self):
                return None
        mdir, proc, slog = None, _Alive(), os.path.join(LOG_DIR, "external_server.log")
        open(slog, "a").close()
        log(f"external server {BASE} (model {SERVED}); skipping install and launch")
    else:
        mdir = model_dir()
        log("model dir:", mdir)
        install_server()
        PHASES["installed"] = round(time.time() - T0)
        preload(mdir)
        PHASES["preloaded"] = round(time.time() - T0)
        proc, slog = launch_server(mdir)
    if not wait_ready(proc, slog, timeout=1800):
        log("server start failed; killing the process tree and retrying once")
        kill_server(proc)
        proc, slog = launch_server(mdir)
        if not wait_ready(proc, slog, timeout=1800):
            raise RuntimeError("server failed to start (twice)")
    PHASES["server_ready"] = round(time.time() - T0)
    if mode == "local":
        if LOCAL_READY_MIN > 0:
            force_at = time.time() + LOCAL_READY_MIN * 60
        else:
            force_at = min(T0 + LOCAL_FORCE_CAP_MIN * 60, max(T0 + LOCAL_FORCE_MIN * 60, time.time() + 20 * 60))
        hard_stop = force_at + LOCAL_HARD_MIN * 60
        log(f"local time box: force at {(force_at - T0) / 60:.0f} min, hard stop {(hard_stop - T0) / 60:.0f} min")
    kv, max_running = server_capacity(slog)
    conc = CONCURRENCY or min(MAX_SEQS, max_running or MAX_SEQS,
                              max(8, int(kv / TOKENS_PER_SEQ_KV)) if kv else 16)
    log(f"concurrency {conc} (max_seqs {MAX_SEQS}, server max running {max_running}, kv {kv})")
    server = {"proc": proc, "down": False, "restarts": 0}

    async def watchdog(force_at):
        """Restart the server (at most twice) if it dies while there is time left to use it."""
        while True:
            await asyncio.sleep(15)
            if server["proc"].poll() is None:
                continue
            server["down"] = True
            log(f"server died (rc={server['proc'].returncode}):\n" + server_log_tail(slog, 40))
            if server["restarts"] >= 2 or force_at - time.time() < 45 * 60:
                log("not restarting the server")
                return
            server["restarts"] += 1
            kill_server(server["proc"])
            server["proc"], _ = launch_server(mdir)
            ok = await asyncio.to_thread(wait_ready, server["proc"], slog, 1800)
            log(f"server restart {'ok' if ok else 'FAILED'}")
            if not ok:
                return
            server["down"] = False

    async def cpu_monitor():
        while True:
            await asyncio.sleep(120)
            try:
                la = os.getloadavg()
                log(f"cpu load avg {la[0]:.1f}/{la[1]:.1f}/{la[2]:.1f} on {os.cpu_count()} cpus")
            except Exception:
                pass

    async def go():
        from concurrent.futures import ThreadPoolExecutor
        # agent_qwen runs executor calls via asyncio.to_thread: one thread per concurrent attempt (E012)
        asyncio.get_running_loop().set_default_executor(ThreadPoolExecutor(max_workers=conc + 8))
        client = Client()
        sch = Scheduler(tasks, sub, client, force_at, hard_stop, conc, attempts=2, tag=mode)
        sch.server_down = lambda: server["down"] or server["proc"].poll() is not None
        wd = asyncio.create_task(watchdog(sch.force_at))
        cm = asyncio.create_task(cpu_monitor())
        await sch.run(order_jobs(tasks))
        wd.cancel()
        cm.cancel()
        sub.write()
        check_submission(OUT_SUB, tasks)
        recs = sch.records
        summary = {"mode": mode, "server": SERVER, "agent": AGENT, "concurrency": conc, "kv_tokens": kv,
                   "phases_s": PHASES, "server_restarts": server["restarts"],
                   "attempts": len(recs), "tokens": sch.tokens,
                   "gen_minutes": round((time.time() - sch.gen_start) / 60, 1),
                   "answered": sum(r["answered"] for r in recs), "errors": sum("error" in r for r in recs),
                   "stream_errors": sum(r.get("stream_errors") or 0 for r in recs),
                   "server_preemption_lines": sum("preempt" in ln.lower() for ln in open(slog, errors="ignore")),
                   "kv_usage_max_pct": max([float(x) for x in re.findall(r"KV cache usage: ([\d.]+)%",
                                                                          open(slog, errors="ignore").read())] or [0]),
                   "tool_calls": sum(r.get("tool_calls") or 0 for r in recs),
                   "tool_errors": sum(r.get("tool_errors") or 0 for r in recs),
                   "parse_failures": sum(r.get("parse_failures") or 0 for r in recs),
                   "answer_sources": {k: sum(r.get("answer_source") == k for r in recs)
                                      for k in ("submit", "forced", "session", "salvage", "parse", "lenient", None)}}
        if solutions:
            sc, n = score(json.load(open(OUT_SUB)), solutions, tasks)
            summary["score"] = f"{sc:.2f}/{n} tasks"
            summary["per_pair"] = [{"task": r["task_id"], "pair": r["pair"], "attempt": r["attempt"],
                                    "correct": r["pred"] == solutions[r["task_id"]][r["pair"]],
                                    "tokens": r["tokens"], "turns": r.get("turns"), "tool_calls": r.get("tool_calls"),
                                    "tool_errors": r.get("tool_errors"), "source": r.get("answer_source"),
                                    "stop": r.get("stop_reason"), "s": r["elapsed_s"], "error": r.get("error")}
                                   for r in recs]
        json.dump(summary, open(os.path.join(WORK, f"summary_{mode}.json"), "w"), indent=1)
        log("SUMMARY", json.dumps(summary))
    asyncio.run(go())
    log("server log tail:\n" + server_log_tail(slog, 10))


if __name__ == "__main__":
    try:
        main()
    except BaseException as exc:  # keep whatever submission.json exists
        import traceback
        traceback.print_exc()
        log(f"FATAL {type(exc).__name__}: {exc}")
    finally:
        try:
            tasks_path = comp_file("arc-agi_test_challenges.json")
            check_submission(os.path.join(WORK, "submission.json"), json.load(open(tasks_path)))
            log("submission.json valid")
        except Exception as exc:
            log(f"submission check failed ({exc}); rewriting fallback")
            OUT_SUB = os.path.join(WORK, "submission.json")
            Submission(json.load(open(comp_file("arc-agi_test_challenges.json")))).write()
        os._exit(0)


## 3. Import check

Every written module must import before any GPU work starts.

In [ ]:
import importlib, sys
sys.path[:0] = ['/kaggle/working/arc_solver', '/kaggle/working/arc_solver/evals']
for m in ['arc_harness.grids', 'arc_harness.arcprize', 'arc_harness.prompts', 'arc_harness.executor', 'arc_harness.agent_qwen', 'solver']:
    mod = importlib.import_module(m)
    print('ok', m, getattr(mod, 'HARNESS_VERSION', ''))

## 4. Run

The solver runs in its own process group. A guard thread kills that group (server and tool children included) at T0 + 11h45m.

In [ ]:
import os, signal, subprocess, sys, threading, time
T0 = float(os.environ['ARC_T0'])
GUARD_AT = T0 + float(os.environ.get('ARC_GUARD_S', 11.75 * 3600))
p = subprocess.Popen([sys.executable, '-u', '/kaggle/working/arc_solver/solver.py'], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working', start_new_session=True)
def guard():
    while p.poll() is None:
        if time.time() > GUARD_AT:
            print('GUARD: wall-clock limit reached; killing the solver process group', flush=True)
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except Exception as exc:
                print('GUARD kill failed', exc, flush=True)
            subprocess.run("pkill -9 -f 'VLLM::'; pkill -9 -f vllm.entrypoints; pkill -9 -f sglang", shell=True)
            return
        time.sleep(10)
threading.Thread(target=guard, daemon=True).start()
for line in p.stdout:
    print(line, end='', flush=True)
print('solver exit code', p.wait(), 'at', round((time.time() - T0) / 3600, 2), 'h')


## 5. Validate `submission.json` (and gate the commit run)

Repairs any invalid slot with the fallback. On non-rerun runs it fails the run if the local validation did not work.

In [ ]:
import glob, json, os, shutil
def _valid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) and 1 <= len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
test_path = (glob.glob('/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json')
             or glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True))[0]
tasks = json.load(open(test_path))
try:
    sub = json.load(open('/kaggle/working/submission.json'))
except Exception as exc:
    print('submission.json unreadable:', exc)
    sub = {}
out, repaired = {}, 0
for tid, t in tasks.items():
    slots = sub.get(tid) if isinstance(sub.get(tid), list) else []
    lst = []
    for i, pair in enumerate(t['test']):
        s = slots[i] if i < len(slots) and isinstance(slots[i], dict) else {}
        fb = pair['input'] if _valid(pair['input']) else [[0]]
        slot = {}
        for k in ('attempt_1', 'attempt_2'):
            ok = _valid(s.get(k))
            repaired += not ok
            slot[k] = s[k] if ok else fb
        lst.append(slot)
    out[tid] = lst
json.dump(out, open('/kaggle/working/submission.json', 'w'))
print(len(out), 'tasks;', sum(len(v) for v in out.values()), 'test inputs; slots repaired:', repaired)
shutil.rmtree('/kaggle/working/pkg', ignore_errors=True)
for f in glob.glob('/kaggle/working/*.tmp'):
    os.remove(f)
# Commit-run gate (never on the rerun; the solver also detects the rerun by the test ids and then writes
# summary_submit.json): the local validation must have worked, else this cell fails the commit.
if os.environ.get('ARC_MODE') != 'submit' and not os.path.exists('/kaggle/working/summary_submit.json'):
    problems = []
    try:
        s = json.load(open('/kaggle/working/summary_local.json'))
    except Exception:
        s = None
        problems.append('summary_local.json missing')
    if s is not None:
        if not s.get('answered'):
            problems.append(f"no answered attempt ({s.get('answered')}/{s.get('attempts')})")
        if s.get('agent') == 'tools' and not s.get('tool_calls'):
            problems.append('no tool calls')
        if s.get('agent') != os.environ.get('ARC_AGENT', 'tools'):
            problems.append('tool agent self-test failed (plain fallback in use)')
    print('COMMIT GATE:', 'PASS' if not problems else 'FAIL: ' + '; '.join(problems))
    if problems:
        raise SystemExit('commit gate failed: ' + '; '.join(problems))
